In [ ]:
#!/usr/bin/env python3
"""
Last ned daglige justerte priser for flyselskap, global indeks og risikofri rente.
Lagrer én CSV per ticker i katalogen som angis i DATA_DIR.
"""

from __future__ import annotations

import pathlib
import yfinance as yf
import pandas as pd

TICKERS = {
    "NAS.OL": "Norwegian Air Shuttle",
    "SASDY": "SAS AB ADR",
    "LHA.DE": "Lufthansa",
    "IAG.L": "International Airlines Group",
    "AF.PA": "Air France-KLM",
    "RYAAY": "Ryanair Holdings",
    "EZJ.L": "easyJet",
    "DAL": "Delta Air Lines",
    "UAL": "United Airlines",
    "LUV": "Southwest Airlines",
}
MARKET_TICKER = "ACWI"     # evt. URTH, IWDA.AS, XDWD.DE
RISK_FREE_TICKER = "^IRX"  # US 13-week T-Bill
DATA_DIR = pathlib.Path("data")

START_DATE = "2019-07-01"
END_DATE = "2020-03-31"

def download_ticker(ticker: str) -> pd.Series:
    df = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=False,
        threads=False,
        repair=True,
    )
    if "Adj Close" not in df or df["Adj Close"].dropna().empty:
        raise ValueError(f"Ingen data for {ticker}")
    return df["Adj Close"]

def main() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    all_symbols = {**TICKERS, MARKET_TICKER: "Global Index", RISK_FREE_TICKER: "Risk Free"}
    failed = {}

    for ticker, name in all_symbols.items():
        try:
            series = download_ticker(ticker)
            out = DATA_DIR / f"{ticker.replace('^','')}.csv"
            series.rename("Adj Close").to_frame().to_csv(out, index_label="Date")
            print(f"[OK] {ticker} -> {out}")
        except Exception as exc:  # pylint:disable=broad-except
            failed[ticker] = str(exc)
            print(f"[FEIL] {ticker}: {exc}")

    if failed:
        print("\nManglet data for:")
        for ticker, msg in failed.items():
            print(f"  {ticker}: {msg}")

if __name__ == "__main__":
    main()
